# మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ — అజ్యూర్ ఓపెన్‌ఐ (ప్రతిస్పందనలు API)

ఈ కోడ్ సాంపిల్‌లో, మీరు **Microsoft Agent Framework (MAF)** ఉపయోగించి **Responses API** ఉపయోగించి **అజ్యూర్ ఓపెన్‌ఐ** ఆధారిత సరీళమైన ఏజెంట్‌ను సృష్టిస్తారు.

> **మైగ్రేషన్ గమనిక:** ఈ సాంపిల్ ముందు Semantic Kernel మరియు GitHub Models ఉపయోగించింది. దీన్ని Microsoft Agent Frameworkకి మార్పిడి చేశారు, మరియు GitHub Models (పట్టుడు పోతున్నవి, 2026 జూలైలో రిటైర్ అవుతున్నవి) ని అజ్యూర్ ఓపెన్‌ఐతో మార్చారు, ఇది Responses APIకి మద్దతు ఇస్తుంది. MAFలో `OpenAIChatClient` అజ్యూర్ ఓపెన్‌ఐ యొక్క స్థిరమైన `/openai/v1/` ఎండ్పాయింట్‌ని లక్ష్యంగా పెట్టుకొని ప్రాథమికంగా Responses APIని ఉపయోగిస్తుంది.

ఈ సాంపిల్ యొక్క ఉద్దేశం తర్వాతి కోడ్ సాంపిల్స్‌లో వివిధ ఏజెంటిక్ నమూనాలను అమలు చేయేటప్పుడు పాటించవలసిన దశలను ప్రదర్శించడం.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## అవసరమైన పైథాన్ ప్యాకేజీలు దిగుమతి చేయండి


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ఒక టూల్‌ను నిర్వచించడం

Microsoft Agent Frameworkలో, ఒక **టూల్** అనేది `@tool` తో అలంకరించబడిన ఒక సాధారణ Python ఫంక్షన్, దాన్ని ఏజెంట్ కాల్ చేయవచ్చు. కింద మేము ఒక టూల్‌ను నిర్వచించాము, ఇది ఒక యాదృచ్ఛిక సెలవు గమ్యస్థానం ఇవ్వడం జరుగుతుంది మరియు గతటువంటి గమ్యస్థానాన్ని మళ్లీ పునరావృతం చేయకుండా జాగ్రత్త వహిస్తుంది.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ఏజెంట్ సృష్టించడం

ఇక్కడ, మేము `TravelAgent` అనే ఏజెంట్‌ను సృష్టిస్తున్నాము.

ఈ ఉదాహరణలో, మేము చాలా సాధారణ సూచనలను ఉపయోగిస్తున్నాము. ఏజెంట్ యొక్క ప్రవర్తన ఎలా మారుతుందో చూడటానికి ఈ సూచనలను స్వేచ్ఛగా మార్చుకోండి.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ఏజెంట్‌ను నడపడం

ఇప్పుడు మనం ఏజెంట్‌ను నడపవచ్చు. ఏజెంట్ సంభాషణను తిప్పుల్లో గుర్తుంచుకోడానికి `AgentSession`ని సృష్టిస్తాము, ఆపై రెండు `user_inputs` పంపుతాము. మొదటిది ఒక ప్రయాణం కోసం అడుగుతుంది; రెండవది యూజర్ సూచనను ఇష్టపడలేదని చెప్పి మరొకదానికి అడుగుతుంది — ఏజెంట్ సెషన్ చరిత్రతో పాటు `get_random_destination` టూల్‌ను ఉపయోగించి స్పందిస్తుంది.

మీరు ఏజెంట్ ఎలా ప్రతిస్పందిస్తుందో చూడటానికి ఈ సందేశాలను మార్చవచ్చు. ప్రతిస్పందనలు **టోకెన్-బై-టోకెన్** విధంగా స్ట్రీమ్ అవుతాయి.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
